In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from os import listdir

import matplotlib.pyplot as plt
from pendulibrary.continuation import adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.plotters import compare_fast
from pendulibrary.integrate import integrate_state
from pendulibrary.common import hamiltonian
from tqdm.auto import tqdm

In [ ]:
fname = "DDsp-P5ish-pre"

data = np.load(f"../database/{fname}.npz")
x0s_in = data["x0s"]
periods_in = data["periods"]
tangents_in = data["tangents"]
eigs_in = data["eigs"]
Lr, Mr = data["params"]

targ = single_fixed(0, x0s_in[0][0], 1, Lr, Mr, 1e-14)
func = targ.g_dg_stm

In [ ]:
# X1 = targ.get_X(x0s_in[29], periods_in[29])
# X2 = targ.get_X(x0s_in[28], periods_in[28])

X1 = targ.get_X(x0s_in[-2], periods_in[-2]/5)
X2 = targ.get_X(x0s_in[-1], periods_in[-1]/5)


g, dg, stm = func(X2)
svd = np.linalg.svd(dg)
svd.S

In [ ]:
np.abs(np.linalg.eigvals(stm))

In [ ]:
tangent = svd.Vh[-1]
sprev = np.linalg.norm(X1 - X2)
if np.dot(tangent, X1 - X2) / sprev < 0:
    tangent *= -1
sprev = np.linalg.norm(X1 - X2)
print(f"Last dist is {sprev:.3e}, dp is {np.dot(tangent, X1 - X2)/sprev:.3f}")

In [ ]:
cont_kwargs = dict(
    s0=sprev / 2.5, s_min=1e-7, S=25.0, tol=1e-11, max_iter=15, target_iter=7, rate=1.05
)
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    X2, func, tangent, **cont_kwargs, exact_tangent=True
)

In [ ]:
fnames = [f.removesuffix(".npz") for f in listdir("../database") if "DDsp" in f]
x0s_new = np.array([targ.get_x0(X) for X in Xs])
periods_new = np.array([targ.get_period(X) for X in Xs])
compare_fast(periods_new, hamiltonian(x0s_new.T, Lr, Mr), fnames)

In [ ]:
np.savez_compressed(
    "../database/DDsp-P5ish-post",
    x0s=x0s_new,
    periods=periods_new,
    eigs=eig_vals,
    tangents=np.column_stack((np.zeros_like(periods_new), tangents)),
    hamiltonians=hamiltonian(x0s_new.T, Lr, Mr),
    params=np.array([Lr, Mr]),
)

In [ ]:
plt.close("all")
ts, xs, fs = integrate_state(x0s_new[-1], periods_new[-1], Lr, Mr, 1e-14)
y = -np.cos(xs[0]) - Lr * np.cos(xs[1])
x = np.sin(xs[0]) + Lr * np.sin(xs[1])

# plt.plot(xs.T)
plt.plot(x, y)
plt.axis("equal")
plt.show()